# Study 928 — Odd-Lot Priority — the teardown

The tape legs and their HAC *t*s, the two round-trip identities, the premium-free breakeven premium and what it still assumes, the sweep over every declared proxy, the era cut with the *contrast* tested, the calendar-time excess-of-cash race and the live synthetic control. Every real number is frozen from `docs/results.md` (Fingerprint `4f4dc9efc748`), 2010-01-28 → 2025-11-21, 167 usable event windows, as-of 2026-06-30.

In [1]:
R = {'start': '2010-01-28', 'end': '2025-11-21', 'n_events': 167, 'n_listed': 178, 'n_filed': 180, 'n_tickers': 128, 'n_ticker_ids': 129, 'fp': '4f4dc9efc748', 'per_year': 10.6, 'premium': 13.0, 'proration': 0.35, 'expiry_td': 21, 'fee': 30.0, 'position': 5000.0, 'cost_bps': 15.0, 'borrow_bps': 40.0, 'runup': 7.41, 'runup_med': 5.94, 'runup_t': 9.45, 'runup_hit': 0.8, 'runup_abn': 6.87, 'runup_abn_t': 9.06, 'drift': 1.44, 'drift_t': 1.75, 'give': -0.5, 'give_t': -1.17, 'give_hit': 0.47, 'perm': 8.63, 'perm_abn': 6.23, 'perm_abn_t': 5.24, 'odd': 5.32, 'odd_t': 7.56, 'hedged': 3.52, 'hedged_t': 4.32, 'hedged_hit': 0.7, 'hedged_lo': 1.95, 'hedged_hi': 5.05, 'round_lot': 1.93, 'round_t': 3.1, 'priority': 3.39, 'priority_t': 4.33, 'prio5': -1.49, 'prio5_t': -1.95, 'prio9': 0.95, 'prio9_t': 1.23, 'prio17': 5.83, 'prio17_t': 7.34, 'prio25': 10.72, 'prio25_t': 13.12, 'prio_zero': 7.4, 'prio_at_be': 1.67, 'prio_at_be_t': 2.15, 'share_post_above_clear': 0.32, 'share_runup_gt_premium': 0.2, 'be_mean': 10.17, 'be_med': 7.93, 'be_p75': 14.56, 'be_share_above': 0.3, 'prem5': -3.99, 'prem5_t': -5.18, 'prem9': -0.23, 'prem9_t': -0.3, 'prem17': 7.27, 'prem17_t': 8.68, 'prem25': 14.78, 'prem25_t': 16.72, 'era_e_n': 74, 'era_e_runup': 5.29, 'era_e_hedged': 5.2, 'era_e_t': 7.13, 'era_e_be': 7.89, 'era_l_n': 93, 'era_l_runup': 9.1, 'era_l_hedged': 2.19, 'era_l_t': 1.72, 'era_l_be': 11.99, 'd_runup': 3.82, 'd_runup_t': 2.63, 'd_be': 4.1, 'd_be_t': 2.49, 'd_hedged': -3.01, 'd_hedged_t': -1.99, 'd_give': -1.39, 'd_give_t': -1.77, 'd_priority': 0.16, 'd_priority_t': 0.11, 'pro15': 4.44, 'pro35': 3.39, 'pro75': 1.31, 'pro100': 0.0, 'fee0': 4.57, 'fee50': 2.37, 'fee50_t': 2.91, 'c100': 0.97, 'c100_t': 1.19, 'c100_be': 12.91, 'c100_fee0': 1.57, 'c100_fee0_t': 1.93, 'cap1000': 1.12, 'cap1000_t': 1.37, 'cap1000_yr': 118, 'cap5000_yr': 1859, 'cap9900_yr': 3990, 'dollar_event': 176, 'borrow300': 3.3, 'borrow300_t': 4.05, 'exp40': 2.42, 'exp40_t': 2.52, 'anchor21': 2.15, 'anchor21_t': 1.98, 'anchor21_be': 22.62, 'cal_days': 4000, 'cal_start': '2010-02-01', 'cal_end': '2025-12-23', 'cal_invested': 51.9, 'cal_live': 1.61, 'cal_vol': 65.6, 'sh_gross': 0.787, 'sh_net': 0.715, 'sh_spy': 0.799, 'sh_adv': -0.084, 'sh_t': 2.09, 'syn_pl_runup': 466, 'syn_pl_runup_t': 10.42, 'syn_pl_give': -387, 'syn_pl_give_t': -9.93, 'syn_nl_runup': -44, 'syn_nl_runup_t': -1.04, 'syn_nl_give': 6, 'syn_nl_give_t': 0.14, 'syn_nl8_runup': 43, 'syn_nl8_sd': 70, 'syn_nl8_fire': 1}

## The estimand, written out

With `p_pre` the close 6 sessions before commencement, `p_entry` the close at *t+1* (the one execution lag), `p_clear = p_pre × (1 + premium)` the assumed clearing price, `p_post` the close 5 sessions after expiry and `f` the round-lot proration fill:

```
1 + r_odd          = (1 + premium) / (1 + runup)
r_odd - r_round    = (1 - f) x [(p_clear - p_post) / p_entry + cost]
breakeven premium  = (p_entry / p_pre) x (1 + r_SPY + frictions) - 1
```

Both identities are asserted in `tests/test_strategy.py`, not just claimed. **Read line 2 carefully:** `p_clear` is the assumed price, so 'what priority is worth' carries the premium assumption exactly as line 1 does — it is not the measured half of this study, and the premium sweep below shows it changing sign. Line 3 is the only one free of the premium.

> 💡 *In plain words:* the premium is a number we assume, the run-up and the post-expiry price are numbers the market printed. The third line asks the tape how big the assumed number would have to be for the trade to be worth doing.

## The tape legs (no premium assumption in the first four)

In [2]:
rows = [('run-up t-6 -> t+1', R['runup'], R['runup_t']),
        ('run-up, abnormal', R['runup_abn'], R['runup_abn_t']),
        ('offer-window drift', R['drift'], R['drift_t']),
        ('post-expiry give-back', R['give'], R['give_t']),
        ('permanent reprice, abnormal', R['perm_abn'], R['perm_abn_t'])]
for label, mean, t in rows:
    flag = '' if abs(t) >= 2 else '   <- not significant'
    print('%-28s mean %+6.2f%%   HAC t %+6.2f%s' % (label, mean, t, flag))
print('\nhit rate on the run-up %.2f  |  on the give-back %.2f (a coin flip)'
      % (R['runup_hit'], R['give_hit']))

run-up t-6 -> t+1            mean  +7.41%   HAC t  +9.45
run-up, abnormal             mean  +6.87%   HAC t  +9.06
offer-window drift           mean  +1.44%   HAC t  +1.75   <- not significant
post-expiry give-back        mean  -0.50%   HAC t  -1.17   <- not significant
permanent reprice, abnormal  mean  +6.23%   HAC t  +5.24

hit rate on the run-up 0.80  |  on the give-back 0.47 (a coin flip)


The give-back is the load-bearing leg and it is **absent**. Odd-lot priority is the right to be paid the tender price instead of holding the stub — worth something only if the stub is worth less. The stub is not worth less: the stock keeps **+6.23%** abnormal permanently, and in **32%** of events `p_post > p_clear`.

## The priced round trips (all four carry the premium) and the premium-free breakeven

In [3]:
print('ALL FOUR of these carry the assumed %.0f%% clearing premium:' % R['premium'])
print('odd lot, net                : %+6.2f%%  HAC t %+5.2f' % (R['odd'], R['odd_t']))
print('odd lot, net + SPY-hedged   : %+6.2f%%  HAC t %+5.2f  boot CI [%+.2f, %+.2f]'
      % (R['hedged'], R['hedged_t'], R['hedged_lo'], R['hedged_hi']))
print('round lot, net (f=0.35)     : %+6.2f%%  HAC t %+5.2f' % (R['round_lot'], R['round_t']))
print('value of odd-lot priority   : %+6.2f%%  HAC t %+5.2f   <- assumption-carried too'
      % (R['priority'], R['priority_t']))
print('\nthe one number free of the premium:')
print('breakeven premium: mean %.2f%%  median %.2f%%  p75 %.2f%%  '
      '(%.0f%% of events need more than the assumed %.0f%%)'
      % (R['be_mean'], R['be_med'], R['be_p75'], R['be_share_above']*100, R['premium']))
print('  ...but it still inherits the anchor (%.2f%% at 21 sessions back), the'
      % R['anchor21_be'])
print('  assumed offer length and the frictions (%.2f%% at a 100 bps touch).'
      % R['c100_be'])

ALL FOUR of these carry the assumed 13% clearing premium:
odd lot, net                :  +5.32%  HAC t +7.56
odd lot, net + SPY-hedged   :  +3.52%  HAC t +4.32  boot CI [+1.95, +5.05]
round lot, net (f=0.35)     :  +1.93%  HAC t +3.10
value of odd-lot priority   :  +3.39%  HAC t +4.33   <- assumption-carried too

the one number free of the premium:
breakeven premium: mean 10.17%  median 7.93%  p75 14.56%  (30% of events need more than the assumed 13%)
  ...but it still inherits the anchor (22.62% at 21 sessions back), the
  assumed offer length and the frictions (12.91% at a 100 bps touch).


## Premium sweep — the assumption that decides the sign of **both** arms

The *t* = +4.3 headline is bought with the premium assumption. So is the *t* = +4.33 attached to 'the value of priority': it is the same assumption wearing a different label, and it crosses zero at 7.4%. The tape only pins the breakeven.

In [4]:
print('assumed   hedged round trip        value of priority')
for p, m, t, pv, pvt in [(5, R['prem5'], R['prem5_t'], R['prio5'], R['prio5_t']),
                         (9, R['prem9'], R['prem9_t'], R['prio9'], R['prio9_t']),
                         (13, R['hedged'], R['hedged_t'], R['priority'], R['priority_t']),
                         (17, R['prem17'], R['prem17_t'], R['prio17'], R['prio17_t']),
                         (25, R['prem25'], R['prem25_t'], R['prio25'], R['prio25_t'])]:
    mark = '  <- default' if p == 13 else ''
    print('%2d%%       %+7.2f%% (t %+6.2f)      %+6.2f%% (t %+6.2f)%s'
          % (p, m, t, pv, pvt, mark))
print('\nhedged round trip crosses zero between 9%% and 13%%;'
      ' value of priority crosses at %.1f%%.' % R['prio_zero'])
print('at the breakeven premium (%.2f%%) priority is worth only %+.2f%% (t %+.2f).'
      % (R['be_mean'], R['prio_at_be'], R['prio_at_be_t']))

assumed   hedged round trip        value of priority
 5%         -3.99% (t  -5.18)       -1.49% (t  -1.95)
 9%         -0.23% (t  -0.30)       +0.95% (t  +1.23)
13%         +3.52% (t  +4.32)       +3.39% (t  +4.33)  <- default
17%         +7.27% (t  +8.68)       +5.83% (t  +7.34)
25%        +14.78% (t +16.72)      +10.72% (t +13.12)

hedged round trip crosses zero between 9% and 13%; value of priority crosses at 7.4%.
at the breakeven premium (10.17%) priority is worth only +1.67% (t +2.15).


## Proration, cost, capacity, borrow — the rest of the assumption surface

In [5]:
print('proration sweep (value of priority) - sets the SIZE; the premium sets the SIGN:')
for f, v in [(0.15, R['pro15']), (0.35, R['pro35']), (0.75, R['pro75']), (1.00, R['pro100'])]:
    print('   fill %.2f -> %+5.2f%%%s' % (f, v, '   <- under-subscribed: worth nothing' if f == 1.0 else ''))
print('   (HAC t is invariant at %+.2f across this whole row - it scales in (1-f))'
      % R['priority_t'])
print('\nflat broker fee on a $%.0f lot: $0 -> %+5.2f%%   $50 -> %+5.2f%% (t=%+.2f)'
      % (R['position'], R['fee0'], R['fee50'], R['fee50_t']))
print('bps touch (charged 3x: buy + SPY hedge round trip), $30 fee:')
print('   15 bps -> %+5.2f%% (t=%+.2f)   100 bps -> %+5.2f%% (t=%+.2f)  '
      '<- micro-cap touch kills it, breakeven %.2f%%'
      % (R['hedged'], R['hedged_t'], R['c100'], R['c100_t'], R['c100_be']))
print('borrow on the short SPY leg  : 300 bps -> %+5.2f%% (t=%+.2f)  [not the binding cost]'
      % (R['borrow300'], R['borrow300_t']))
print('offer length 40 td           : %+5.2f%% (t=%+.2f)' % (R['exp40'], R['exp40_t']))
print('anchor 21 sessions back      : %+5.2f%% (t=%+.2f), breakeven %.2f%%  <- the most fragile choice'
      % (R['anchor21'], R['anchor21_t'], R['anchor21_be']))
print('\ncapacity (a 99-share lot is 99 x the share price):')
for pos, m, t, yr in [(1000, R['cap1000'], R['cap1000_t'], R['cap1000_yr']),
                      (5000, R['hedged'], R['hedged_t'], R['cap5000_yr']),
                      (9900, 3.82, 4.68, R['cap9900_yr'])]:
    print('   $%5d lot -> %+5.2f%% (t=%+.2f)  ~$%s per year' % (pos, m, t, format(yr, ',')))

proration sweep (value of priority) - sets the SIZE; the premium sets the SIGN:
   fill 0.15 -> +4.44%
   fill 0.35 -> +3.39%
   fill 0.75 -> +1.31%
   fill 1.00 -> +0.00%   <- under-subscribed: worth nothing
   (HAC t is invariant at +4.33 across this whole row - it scales in (1-f))

flat broker fee on a $5000 lot: $0 -> +4.57%   $50 -> +2.37% (t=+2.91)
bps touch (charged 3x: buy + SPY hedge round trip), $30 fee:
   15 bps -> +3.52% (t=+4.32)   100 bps -> +0.97% (t=+1.19)  <- micro-cap touch kills it, breakeven 12.91%
borrow on the short SPY leg  : 300 bps -> +3.30% (t=+4.05)  [not the binding cost]
offer length 40 td           : +2.42% (t=+2.52)
anchor 21 sessions back      : +2.15% (t=+1.98), breakeven 22.62%  <- the most fragile choice

capacity (a 99-share lot is 99 x the share price):
   $ 1000 lot -> +1.12% (t=+1.37)  ~$118 per year
   $ 5000 lot -> +3.52% (t=+4.32)  ~$1,859 per year
   $ 9900 lot -> +3.82% (t=+4.68)  ~$3,990 per year


## Era cut (split 2018-01-01, fixed in advance) — and the contrast, tested

Two point estimates are not a contrast. The desk rule is that a sub-period claim is a claim about a *difference*, so the difference gets its own test (Welch *t* on the per-event vectors, late minus early).

In [6]:
print('2010-2017 (n=%d): run-up %+5.2f%%  hedged %+5.2f%% (t=%+.2f)  breakeven %5.2f%%'
      % (R['era_e_n'], R['era_e_runup'], R['era_e_hedged'], R['era_e_t'], R['era_e_be']))
print('2018-2025 (n=%d): run-up %+5.2f%%  hedged %+5.2f%% (t=%+.2f)  breakeven %5.2f%%   <- t < 2'
      % (R['era_l_n'], R['era_l_runup'], R['era_l_hedged'], R['era_l_t'], R['era_l_be']))
print('\nthe CHANGE itself (Welch t, late - early):')
for lbl, gap, t in [('run-up', R['d_runup'], R['d_runup_t']),
                    ('breakeven premium', R['d_be'], R['d_be_t']),
                    ('hedged round trip', R['d_hedged'], R['d_hedged_t']),
                    ('give-back', R['d_give'], R['d_give_t']),
                    ('value of priority', R['d_priority'], R['d_priority_t'])]:
    flag = 'real change' if abs(t) >= 2 else 'NOT a significant change'
    print('   %-18s gap %+6.2f pp   Welch t %+5.2f   %s' % (lbl, gap, t, flag))
print('\n=> what demonstrably changed is the ENTRY PRICE, not the profit:')
print('   the decay of the round trip is itself only t = %+.2f.' % R['d_hedged_t'])

2010-2017 (n=74): run-up +5.29%  hedged +5.20% (t=+7.13)  breakeven  7.89%
2018-2025 (n=93): run-up +9.10%  hedged +2.19% (t=+1.72)  breakeven 11.99%   <- t < 2

the CHANGE itself (Welch t, late - early):
   run-up             gap  +3.82 pp   Welch t +2.63   real change
   breakeven premium  gap  +4.10 pp   Welch t +2.49   real change
   hedged round trip  gap  -3.01 pp   Welch t -1.99   NOT a significant change
   give-back          gap  -1.39 pp   Welch t -1.77   NOT a significant change
   value of priority  gap  +0.16 pp   Welch t +0.11   NOT a significant change

=> what demonstrably changed is the ENTRY PRICE, not the profit:
   the decay of the round trip is itself only t = -1.99.


## Calendar-time sleeve, excess-of-cash — the un-achievable scaled version

Equal-weight across live positions, BIL when idle, both arms minus BIL. This applies the odd-lot **fill** assumption at portfolio scale — which is exactly what the odd-lot rule forbids — *and* it carries the assumed 13% clearing premium, so its positive mean is not evidence for anything. It is priced only to close the door.

> 💡 *In plain words:* even if the law let you do this trade with real money, it would not have been a better use of the money than owning the index.

In [7]:
print('%s -> %s, %d sessions, invested %.1f%% of days, %.2f live names, vol %.1f%%'
      % (R['cal_start'], R['cal_end'], R['cal_days'], R['cal_invested'],
         R['cal_live'], R['cal_vol']))
print('sleeve excess Sharpe  gross %+.3f  net %+.3f' % (R['sh_gross'], R['sh_net']))
print('SPY    excess Sharpe        %+.3f      -> advantage %+.3f (HAC t on the daily diff %+.2f)'
      % (R['sh_spy'], R['sh_adv'], R['sh_t']))
print('higher mean, lower Sharpe: 1.6 single names at full weight is a 66%-vol sleeve.')

2010-02-01 -> 2025-12-23, 4000 sessions, invested 51.9% of days, 1.61 live names, vol 65.6%
sleeve excess Sharpe  gross +0.787  net +0.715
SPY    excess Sharpe        +0.799      -> advantage -0.084 (HAC t on the daily diff +2.09)
higher mean, lower Sharpe: 1.6 single names at full weight is a 66%-vol sleeve.


## Live synthetic control — the machinery is unbiased

**Synthetic data**, not the tender list. Planted: a +500 bps run-up and a −400 bps post-expiry give-back, both of which must be recovered. Null: nothing planted, nothing may be found.

In [8]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
from odd_lot import data, strategy as st
# SYNTHETIC tape only - not the real tender list.
panel_p, ev_p, truth = data.synthetic_panel(n_events=90, signal_strength=1.0, seed=928)
panel_0, ev_0, _     = data.synthetic_panel(n_events=90, signal_strength=0.0, seed=928)
pl = st.synthetic_detect(panel_p, ev_p)
nl = st.synthetic_detect(panel_0, ev_0)
print('planted %+d bps run-up / %+d bps give-back ->  recovered %+.0f bps (t=%+.2f) / '
      '%+.0f bps (t=%+.2f)'
      % (truth['runup_planted_bps'], -truth['giveback_planted_bps'],
         pl['runup_bps'], pl['runup_t'], pl['giveback_bps'], pl['giveback_t']))
print('null (nothing planted)            ->  recovered %+.0f bps (t=%+.2f) / '
      '%+.0f bps (t=%+.2f)'
      % (nl['runup_bps'], nl['runup_t'], nl['giveback_bps'], nl['giveback_t']))

planted +500 bps run-up / -400 bps give-back ->  recovered +512 bps (t=+9.71) / -376 bps (t=-7.67)
null (nothing planted)            ->  recovered -1 bps (t=-0.02) / +17 bps (t=+0.33)


In [9]:
import numpy as np
nulls = [st.synthetic_detect(*data.synthetic_panel(n_events=60,
                                                   signal_strength=0.0,
                                                   seed=928 + s)[:2])
         for s in range(6)]
ru = np.array([n['runup_bps'] for n in nulls])
gb = np.array([n['giveback_bps'] for n in nulls])
print('null x6: run-up mean %+.0f bps (sd %.0f), |t|>=2 in %d/6'
      % (ru.mean(), ru.std(ddof=1), sum(abs(n['runup_t']) >= 2 for n in nulls)))
print('null x6: give-back mean %+.0f bps, |t|>=2 in %d/6'
      % (gb.mean(), sum(abs(n['giveback_t']) >= 2 for n in nulls)))

null x6: run-up mean +28 bps (sd 88), |t|>=2 in 1/6
null x6: give-back mean +42 bps, |t|>=2 in 0/6


## Verdict

- **Signal — Mixed**, split by leg: **Real on the announcement run-up · None on the give-back the claim depends on.** The run-up is **+7.41%** (abnormal +6.87%, HAC *t* = +9.06) with no proxy anywhere in it; the post-expiry give-back is **-0.50%**, *t* = -1.17, median 0.00%. **Every** profitable number in the study is the assumed 13% clearing premium meeting the tape — the hedged round trip **+3.52%** (*t* = +4.32) *and* the 'value of priority' **+3.39%** (*t* = +4.33) alike; they cross zero at 9-13% and 7.4% respectively. Premium-free, the tape says the breakeven is **10.17%**, **11.99%** post-2018 (the rise is itself significant, Welch *t* = +2.49) and **12.91%** at a micro-cap 100 bps touch. The synthetic control recovers a planted run-up (+466 bps of 500) and a planted give-back (-387 bps of −400) and is silent on the null (-44 bps, *t* = -1.04) — those are `docs/results.md`'s 150-event panels; the live cells above run a 90-event one and land in the same place — so the missing give-back is a fact about tenders, not a broken harness.
- **Tradability — Mirage.** The entitlement caps the position at 99 shares: **$176 per event**, **~$1,859 a year** at a $50 share price and **$118** at a $10 one, where the flat fee is 300 bps and *t* = +1.37. The scaled fantasy sleeve loses the excess-of-cash Sharpe race to SPY (+0.715 vs +0.799) at 66% vol. Nothing bankable at desk size.